In [9]:
#introduce Train and Test Set Again
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

df_model_ready = pd.read_csv('Cleaned_California_Housing.csv', low_memory=False)


def prepare_oot_modeling_data(df, target_col, x_months):
    df['CloseDate'] = pd.to_datetime(df['CloseDate'])
    max_date = df['CloseDate'].max()
    test_start_date = max_date - pd.DateOffset(months=1)
    train_start_date = test_start_date - pd.DateOffset(months=x_months)

    test_mask = (df['CloseDate'] > test_start_date) & (df['CloseDate'] <= max_date)
    train_mask = (df['CloseDate'] > train_start_date) & (df['CloseDate'] <= test_start_date)

    df_train = df[train_mask].copy()
    df_test = df[test_mask].copy()

    y_train = np.log1p(df_train[target_col]) 
    y_test = np.log1p(df_test[target_col])

    X_train = df_train.drop(columns=[target_col, 'CloseDate'])
    X_test = df_test.drop(columns=[target_col, 'CloseDate'])

    num_cols = ['LivingArea', 'LotSizeSquareFeet', 'BedroomsTotal', 
                'BathroomsTotalInteger', 'YearBuilt', 'AssociationFee', 'GarageSpaces']

    scaler = StandardScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols] = scaler.transform(X_test[num_cols])

    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = prepare_oot_modeling_data(df_model_ready, 'ClosePrice', 12)

In [10]:
#Decision Tree
dt_model = DecisionTreeRegressor(random_state=42)

dt_model.fit(X_train, y_train)


dt_train_pred = dt_model.predict(X_train)
dt_test_pred = dt_model.predict(X_test)


train_r2 = r2_score(y_train, dt_train_pred)
test_r2 = r2_score(y_test, dt_test_pred)

print(f"Train R²: {train_r2:.4f}")
print(f" Test R²: {test_r2:.4f}")
print(f"Compare to  Baseline (Train R²): 0.7254")

Train R²: 0.9989
 Test R²: 0.6162
Compare to  Baseline (Train R²): 0.7254


This model has serious generalization issues and is overfitted. I need to try tuning the parameters to constrain the model, starting with limiting the depth of the tree.

In [11]:
dt_tuned_model = DecisionTreeRegressor(max_depth=12, random_state=12)

dt_tuned_model.fit(X_train, y_train)


dt_tuned_train_pred = dt_tuned_model.predict(X_train)
dt_tuned_test_pred = dt_tuned_model.predict(X_test)

tuned_train_r2 = r2_score(y_train, dt_tuned_train_pred)
tuned_test_r2 = r2_score(y_test, dt_tuned_test_pred)

print(f" Train R²: {tuned_train_r2:.4f}")
print(f" Test R²: {tuned_test_r2:.4f}")
print(f" Limit-Free Tree Test R²: 0.6162")
print(f" Baseline Test R²: 0.7254")

 Train R²: 0.6609
 Test R²: 0.6293
 Limit-Free Tree Test R²: 0.6162
 Baseline Test R²: 0.7254


Although this result avoids overfitting, it reduces the model’s accuracy somewhat—even more than the linear model. Since parameter tuning cannot improve the model’s accuracy at this point, we’ll have to use a random forest model.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)


rf_model.fit(X_train, y_train)

rf_train_pred = rf_model.predict(X_train)
rf_test_pred = rf_model.predict(X_test)

rf_train_r2 = r2_score(y_train, rf_train_pred)
rf_test_r2 = r2_score(y_test, rf_test_pred)


print(f" Train R²: {rf_train_r2:.4f}")
print(f" Test R²: {rf_test_r2:.4f}")
print(f"Compare to Baseline Test R²: 0.7254")


 Train R²: 0.7540
 Test R²: 0.7079
Compare to Baseline Test R²: 0.7254


In this housing price prediction study using out-of-time validation, we observed a classic phenomenon: the test set performance of the Random Forest model (R² = 0.7079) failed to outperform the baseline linear regression model (R² = 0.7254).
I speculate that a possible reason is that linear regression has strong extrapolation capabilities and performs well with geocoded data, whereas the Random Forest model does not handle such sparse, high-dimensional features well. Furthermore, the Random Forest model is not well-suited for extrapolation. In future work, we could try using hybrid models or apply specialized feature engineering to the Random Forest model.